# Brazil data validation - INEP School Census + Attainment Rates

**Living Stone Foundation - Applied Data Lab**

**Who this notebook is for:** readers who did **not** build the project.
**Goal:** prove that the population (Brazilian basic-education **schools**) matches the labels we train on (official INEP **dropout rates**).


## Variable dictionary (read before the charts)

| Column / label in charts | Plain-English meaning | How to read values |
|---|---|---|
| `target_dropout_rate` / **Dropout rate (%)** | Official INEP "taxa de abandono": share of students who **stopped attending** during the school year, after the Census reference date | 0 = nobody left; 5 = 5% left. Higher = worse |
| `year` | School Census / attainment-rate year | Years present in the mart (currently 2018-2025) |
| `school_id` (`CO_ENTIDADE`) | Unique school code (INEP) | Join key between Census and attainment-rate data |
| `state_code` | Brazilian state abbreviation ("UF" = Unidade Federativa) | e.g. SP, BA, AM |
| `municipality_id` | IBGE municipality code | Geographic context |
| `admin_dependency_type` | Administrative network | 1=Federal, 2=State, 3=Municipal, 4=Private |
| `location_type` | School location type | 1=Urban, 2=Rural |
| `is_rural` | 1 if rural school | Shortcut of `location_type == 2` |
| `is_public` | 1 if public network (federal/state/municipal) | 0 = private |
| `has_water` | Has potable / public water | 1=yes, 0=no |
| `has_electricity` | Connected to public electricity | 1=yes, 0=no |
| `has_sewage` | Public sewage connection | 1=yes, 0=no |
| `has_internet` / `has_internet_for_students` | Internet at school / specifically available to students | 1=yes, 0=no |
| `has_library` | Library / reading room | 1=yes, 0=no |
| `has_computer_lab` | Computer lab | 1=yes, 0=no |
| `has_sports_court` | Sports court | 1=yes, 0=no |
| `enrollment_basic_ed` | Enrollment count (basic education total, when available) | Larger = bigger school |
| `enrollment_level` | Enrollment used for this level (Fundamental or Medio) | Filter requires >= 20 students |
| `teacher_count_basic_ed` | Number of teachers (basic education) | Staffing intensity |
| `student_teacher_ratio` | Students / teachers | Higher often means more crowded classes |
| `avg_class_size` | Enrollment / number of classes for this level | Higher = larger classes |
| `overage_enrollment_share` | Share of students above the expected age for this level | A known proxy for grade-age distortion, itself a dropout correlate |
| `dropout_rate_lag1` / `_lag2` | This school's own dropout rate 1 / 2 years earlier | Missing (NaN) for a school's first year in the panel |
| `dropout_rate_3yr_avg` / `_trend` | Trailing 3-year average / (lag1 - lag2) | Trend > 0 means the rate has been rising |
| `has_history` | 1 if the school has at least one prior year on record | 0 = first appearance in the 2018-2025 window |
| `approval_rate_lag1` / `failure_rate_lag1` | This school's own approval / grade-repetition rate, prior year | A rising failure rate is a known leading indicator of dropout |
| `municipal_dropout_rate_lag1` / `state_dropout_rate_lag1` | Average dropout rate across the municipality / state, prior year | A spatial-lag signal for local shocks a single school's history cannot capture |
| `is_covid_year` | 1 for school years 2020-2021 | INEP suspended normal grade-progression rules in these years |
| `risk_band` | low / moderate / high | Relative triage label inside the level |
| `high_risk` | 1 if school is in the elevated-risk group | Binary flag used for the triage/ranking metrics |
| **MAE / RMSE / R2** | Model error metrics (later notebooks / model folder) | Lower MAE/RMSE better; R2 closer to 1 better |

### Acronyms
| Acronym | Meaning |
|---|---|
| **INEP** | Brazil's federal education-statistics agency |
| **School Census** ("Censo Escolar") | Annual school census (structure + enrollment) |
| **School Attainment Rates** ("Taxas de Rendimento") | Official approval / failure / dropout rates |
| **EDA** | Exploratory Data Analysis |
| **UF** | Unidade Federativa (Brazilian state) |
| **Fundamental** | Ensino Fundamental: grades 1-9, roughly ages 6-14 |
| **Medio** | Ensino Medio: grades 10-12, Brazil's upper-secondary stage |
| **EJA** | Educacao de Jovens e Adultos (adult / youth education track) |


### How to read this notebook
1. Run cells top to bottom (`Shift+Enter`).
2. After every code cell, read the **Insight** markdown - that is the takeaway, not the raw JSON.


In [1]:
# Cell A - project path only (no Path.cwd / exists / resolve)
import sys
ROOT = r"C:\Users\User\Desktop\Projeto Living Stone Foundation"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


### What this cell did
Added the project folder to Python's import path using a **fixed string** (no `Path.cwd()` / `exists` / `resolve`). Those path checks can freeze kernels on Desktop/OneDrive.


In [2]:
# Cell B - imports (first run can take a minute for pandas/seaborn)
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

FEATURE_LABELS = {
    "is_rural": "Rural school (1=yes)",
    "is_public": "Public network (1=yes)",
    "has_internet": "Has internet (1=yes)",
    "has_internet_for_students": "Has internet for students (1=yes)",
    "has_computer_lab": "Has computer lab (1=yes)",
    "has_sports_court": "Has sports court (1=yes)",
    "has_library": "Has library (1=yes)",
    "has_water": "Has water (1=yes)",
    "has_electricity": "Has electricity (1=yes)",
    "has_sewage": "Has sewage (1=yes)",
    "enrollment_level": "Enrollment (this level)",
    "enrollment_basic_ed": "Basic-ed enrollment (total)",
    "teacher_count_basic_ed": "Number of teachers",
    "student_teacher_ratio": "Students per teacher",
    "avg_class_size": "Average class size",
    "overage_enrollment_share": "Share of over-age students",
    "admin_dependency_type": "Admin network code",
    "location_type": "Urban/rural code",
    "target_dropout_rate": "Dropout rate (%)",
    "dropout_rate_lag1": "Dropout rate, prior year",
    "dropout_rate_lag2": "Dropout rate, 2 years prior",
    "dropout_rate_3yr_avg": "Dropout rate, 3-year trailing avg",
    "dropout_rate_trend": "Dropout rate trend",
    "approval_rate_lag1": "Approval rate, prior year",
    "failure_rate_lag1": "Failure rate, prior year",
    "municipal_dropout_rate_lag1": "Municipality's dropout rate, prior year",
    "state_dropout_rate_lag1": "State's dropout rate, prior year",
}
import json
from src.eda import compare_levels, validation_json_path
print("Imports OK")


Imports OK


### What this cell did
Loaded charting/table libraries. The **first** run can take ~30-90s while pandas/seaborn warm up - that is normal, not a freeze on `import sys`.


## 1. Official validation artifact


In [3]:
path = validation_json_path()
payload = json.loads(path.read_text(encoding='utf-8'))
print('File:', path)
print('Source:', payload.get('source'))
print('Dropout-rate definition:', payload.get('definition_dropout_rate'))
print('Grain:', payload.get('grain'))
print('Years:', payload.get('years'))
print('Rows / schools:', payload.get('rows'), '/', payload.get('schools'))
print('Coverage:', json.dumps(payload.get('coverage'), indent=2))
print('Student-level labels available?', payload.get('student_level_labeled_dropout_available'))


File: C:\Users\User\Desktop\Projeto Living Stone Foundation\latam_education_data\dq\rendimento_validation.json
Source: INEP Taxas de Rendimento Escolar (official)
Dropout-rate definition: Percentage of students who stopped attending school after the School Census reference date during the school year (student movement status = left attending). Computed by INEP from the School Census 'Situacao do Aluno' module.
Grain: school x year (school_id = CO_ENTIDADE)
Years: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Rows / schools: 1040730 / 147330
Coverage: {
  "fundamental_non_null_pct": 0.925817455055586,
  "medio_non_null_pct": 0.22405907391926821,
  "fundamental_mean": 1.203613394968065,
  "medio_mean": 3.324452258936038,
  "approval_rate_fundamental_non_null_pct": 0.925817455055586,
  "failure_rate_fundamental_non_null_pct": 0.925817455055586
}
Student-level labels available? False


### How to read this output
- **Source** should cite INEP School Attainment Rates (official), not a homemade proxy.
- **Definition** means: students who *left attending* during the year - not "failed the year" and not "never enrolled".
- **Grain = school x year** means each measurement is about a **school**, not a named student.
- **Coverage**
  - `fundamental_non_null_pct` near ~90%+: most schools have a Fundamental dropout-rate number.
  - `medio_non_null_pct` much lower (~20%+): only schools that offer Ensino Medio report that rate.
  - Means: Medio dropout rate is typically **higher** than Fundamental (e.g. ~3% vs ~1%).
- **Student-level labels available? -> False**: we cannot honestly build a student scorer with these open files.

### Insight (decision for the project)
Population and labels are aligned for a **Brazil school-level** early-warning system. They are **not** aligned with the old Kaggle higher-education student dataset - that is why it was removed (see `docs/scope_revision.md`).


## 2. Side-by-side mart summary (Fundamental vs Medio)


In [4]:
summary = compare_levels()
# Friendlier column names for readers
pretty = summary.rename(columns={
    'level': 'Education level',
    'rows': 'School-year rows',
    'schools': 'Unique schools',
    'mean_dropout_rate': 'Mean dropout rate (%)',
    'median_dropout_rate': 'Median dropout rate (%)',
    'p90_dropout_rate': '90th percentile dropout rate (%)',
    'public_share': 'Share public schools',
    'rural_share': 'Share rural schools',
    'share_with_history': 'Share with prior-year history',
})
display(pretty)


,Education level,School-year rows,Unique schools,Mean dropout rate (%),Median dropout rate (%),90th percentile dropout rate (%),Share public schools,Share rural schools,Share with prior-year history
0,fundamental,832326,122211,1.142,0.000,3.300,0.799,0.313,0.856
1,medio,225199,32612,3.314,0.500,10.200,0.722,0.104,0.858


### How to read this table
Each row is one education level after joining Census features to official dropout rates and keeping schools with enough enrollment (>= 20).

| Column | Takeaway question |
|---|---|
| School-year rows | Do we have enough data to model? |
| Mean vs median dropout rate | Is the outcome rare/skewed? (median near 0 with mean > median means many zeros + a right tail) |
| 90th percentile | What does a "high" school look like in this level? |
| Public / rural shares | What kind of schools dominate the sample? |
| Share with prior-year history | How much of the data can use the lag/history features (see notebooks 02/03)? |

### Insight
If **Medio** shows higher mean dropout rate and a heavier right tail than **Fundamental**, training **two models** (and two Streamlit apps) is justified: the risk regimes are different. A single pooled model would mix two different populations.
